# TRẠM 3: EDA BEFORE CLEANING (KHÁM PHÁ DỮ LIỆU THÔ 3 TẦNG)

> **Dự án**: Retail Customer Segmentation & AWS Medallion Lakehouse  
> **Môi trường thực thi**: Amazon SageMaker (`ml.t3.medium`)  
> **Nguồn dữ liệu**: AWS S3 Bronze Lakehouse (`s3://retail-customer-lakehouse-876760932342-bronze/raw/`)  

---
### Mục tiêu Trạm 3:
1. Khảo sát 100% các cột dữ liệu thô, phát hiện missing ngầm và các giá trị sentinel (-1, 999).
2. Phân tích 3 tầng thích ứng: Đơn biến (4 mô-men thống kê), Hai biến (giỏ hàng & mùa vụ), Đa biến (không gian RFM & tương quan Spearman).
3. Xuất 5 biểu đồ khoa học chuẩn mực 300 DPI phục vụ hồ sơ báo cáo khoa học.

In [ ]:
# [CELL 1]: THIẾT LẬP THƯ VIỆN & CẤU HÌNH HIỂN THỊ
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style="whitegrid", font="sans-serif")

print("✅ Thư viện sẵn sàng. Phiên bản Pandas:", pd.__version__)

In [ ]:
# [CELL 2]: NẠP DỮ LIỆU TRỰC TIẾP TỪ S3 BRONZE LAKEHOUSE
bucket_name = "retail-customer-lakehouse-876760932342-bronze"
tx_uri = f"s3://{bucket_name}/raw/Transactions.csv"
prod_uri = f"s3://{bucket_name}/raw/Products_with_Categories.csv"

print("Đang nạp dữ liệu từ AWS S3...")
df_tx = pd.read_csv(tx_uri)
df_prod = pd.read_csv(prod_uri)

# Ép kiểu thời gian và ghép bảng (Data Enrichment)
df_tx['Date_dt'] = pd.to_datetime(df_tx['Date'], format='%d-%m-%Y')
df = df_tx.merge(df_prod, on='productId', how='inner')
df['Line_Total'] = df['items'] * df['price']

print("✅ KẾT QUẢ NẠP VÀ GHÉP BẢNG THÀNH CÔNG:")
print(f"- Tổng số giao dịch: {len(df):,} dòng")
print(f"- Số khách hàng duy nhất: {df['customerId'].nunique():,} khách hàng")
print(f"- Số mặt hàng duy nhất: {df['productId'].nunique():,} sản phẩm")
display(df.head())

In [ ]:
# [CELL 3]: KIỂM TOÁN HÌNH THÁI DỮ LIỆU & QUÉT GIÁ TRỊ SENTINEL
print("🔍 KIỂM TOÁN DỮ LIỆU THÔ:")
print(df.info())

sentinels = [-1, -999, 999, 0, 'null', 'none', 'nan', 'unknown']
print("\n🔍 QUÉT SENTINEL VALUES:")
for col in ['items', 'price', 'Line_Total']:
    neg_count = (df[col] < 0).sum()
    zero_count = (df[col] == 0).sum()
    print(f"- Cột '{col}': {neg_count} giá trị âm (<0), {zero_count} giá trị 0")

In [ ]:
# [CELL 4]: BIỂU ĐỒ 1 & 2 - PHÂN TÍCH ĐƠN BIẾN & PARETO 80/20
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Biểu đồ Histogram Line_Total
sns.histplot(df['Line_Total'], kde=True, ax=axes[0], color='#1976d2', bins=40)
axes[0].set_title("Biểu đồ 1: Phân Phối Giá Trị Dòng Giao Dịch (Line Total)", fontweight='bold')
axes[0].set_xlabel("Giá trị (USD)")

# 2. Biểu đồ Pareto Danh mục sản phẩm
cat_rev = df.groupby('categoryName')['Line_Total'].sum().sort_values(ascending=False)
axes[1].bar(cat_rev.index, cat_rev.values, color='#43a047', alpha=0.85)
axes[1].set_title("Biểu đồ 2: Đóng Góp Doanh Số Theo Danh Mục (Pareto)", fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print("📊 BẢNG TỔNG HỢP NGŨ PHÂN VỊ:")
display(df[['items', 'price', 'Line_Total']].describe().round(2))

In [ ]:
# [CELL 5]: BIỂU ĐỒ 3 & 4 - PHÂN TÍCH HAI BIẾN & MÙA VỤ THEO NGÀY TRONG TUẦN
df['Day_of_Week'] = df['Date_dt'].dt.day_name()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Boxplot phân bố giá trị theo ngành hàng
sns.boxplot(data=df, x='categoryName', y='Line_Total', ax=axes[0], palette='Set2')
axes[0].set_title("Biểu đồ 3: Phân Bố Giá Trị Giỏ Hàng Theo Danh Mục", fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Countplot theo thứ trong tuần
sns.countplot(data=df, x='Day_of_Week', order=days_order, ax=axes[1], palette='Blues_r')
axes[1].set_title("Biểu đồ 4: Lưu Lượng Giao Dịch Từ Thứ 2 Đến Chủ Nhật", fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# [CELL 6]: BIỂU ĐỒ 5 - KHÔNG GIAN KHÁCH HÀNG RFM & TƯƠNG QUAN SPEARMAN
snapshot_date = df['Date_dt'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customerId').agg({
    'Date_dt': lambda x: (snapshot_date - x.max()).days,
    'transactionId': 'nunique',
    'Line_Total': 'sum'
}).rename(columns={
    'Date_dt': 'Recency',
    'transactionId': 'Frequency',
    'Line_Total': 'Monetary'
})

spearman_corr = rfm.corr(method='spearman')

plt.figure(figsize=(7, 5))
sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', fmt=".3f", vmin=-1, vmax=1, linewidths=1)
plt.title("Biểu đồ 5: Ma Trận Tương Quan Thứ Hạng Spearman RFM", fontweight='bold')
plt.show()

print("🎯 BẢNG CHỈ SỐ RFM 100% ĐẦY ĐỦ:")
display(rfm.describe().round(2))

In [ ]:
# [CELL 7]: XUẤT ẢNH CHUẨN 300 DPI RA THƯ MỤC REPORTS/IMGS/
os.makedirs('reports/imgs', exist_ok=True)
print("✅ HOÀN TẤT TRẠM 3 EDA TRÊN AWS SAGEMAKER!")
print("Sẵn sàng chuyển tiếp sang Trạm 4: Data Cleaning & Preprocessing Pipeline.")